In [ ]:
!rm -rf logs/ # clear logs
!rm -rf optimizer_output/

# Imports

In [ ]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice again
!ngspice -v

In [ ]:
import sympy as sp
import logging

from pathlib import Path

from symxplorer.spice_engine             import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools           import Nevergrad_Spice_Multi_Spec_Optimizer, Project_Setup

from symxplorer.logging import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

# Instantiations


In [ ]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

In [ ]:
# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

In [ ]:
# (2) Create the Spice Simulator Wrapper
wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename= PROJECT_SETUP.ws_root / PROJECT_SETUP.netlist,
    output_folder=PROJECT_SETUP.ws_root / PROJECT_SETUP.outdir,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

In [ ]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Optimizer(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

## Sanity Check

In [ ]:
wrapper.run_sanity_check(
    use_editor=True,
    sim_execution_t=Sim_Execution_Type.RUN_NOW
)

# Method Calls

In [ ]:
circuit_optimizer.parameterize()

In [ ]:
circuit_optimizer.optimize()

In [ ]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

In [ ]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

In [ ]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

# Testing

In [ ]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

In [ ]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

In [ ]:
circuit_optimizer.compute_spec_loss(spec_curr_val=-90, target_spec=target_spec)

In [ ]:
PROJECT_SETUP.dut_params